In [0]:
#LLM-Powered Survey Theme Extraction


In [0]:
#Config + sample
CATALOG = "cx"
SCHEMA_SILVER = "cx_silver"
SCHEMA_GOLD = "cx_gold"

# Choose your LLM model — change if needed
LLM_MODEL = "databricks-meta-llama-3-3-70b-instruct"

# Quick check — how many comments do we have to classify?
surveys = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.surveys")
comments_count = surveys.filter("open_comment IS NOT NULL AND open_comment != ''").count()
print(f"Comments to classify: {comments_count:,}")
print(f"Using model: {LLM_MODEL}")

Comments to classify: 23,384
Using model: databricks-meta-llama-3-3-70b-instruct


In [0]:
%sql
--Test ai_query on 5 comments
SELECT
  open_comment,
  ai_query(
    'databricks-meta-llama-3-3-70b-instruct',
    CONCAT(
      'Classify this customer survey comment into ONE theme from this list. ',
      'Return only the theme name, no explanation. ',
      'Themes: reliability, support_quality, training, pricing, software, other. ',
      'Comment: ', open_comment
    )
  ) AS theme
FROM cx.cx_silver.surveys
WHERE open_comment IS NOT NULL AND open_comment != ''
LIMIT 5;

open_comment,theme
Software updates have introduced new bugs. Reliability is a major concern.,reliability
Too expensive for what we get. Considering switching vendors.,pricing
Too expensive for what we get. Considering switching vendors.,pricing
Consumables are high quality and shipping is consistent.,reliability
Consumables are high quality and shipping is consistent.,reliability


In [0]:
%sql
--Classify all comments and save to Gold
CREATE OR REPLACE TABLE cx.cx_gold.survey_themes AS
SELECT
  survey_id,
  customer_id,
  survey_date,
  nps_score,
  nps_bucket,
  open_comment,
  LOWER(TRIM(
    ai_query(
      'databricks-meta-llama-3-3-70b-instruct',
      CONCAT(
        'Classify this customer survey comment into ONE theme. ',
        'Return only the theme name in lowercase, no explanation. ',
        'Choose from: reliability, support_quality, training, pricing, software, other. ',
        'Comment: ', open_comment
      )
    )
  )) AS theme,
  LOWER(TRIM(
    ai_query(
      'databricks-meta-llama-3-3-70b-instruct',
      CONCAT(
        'What is the sentiment of this customer comment? ',
        'Return ONLY one word: positive, negative, or neutral. ',
        'Comment: ', open_comment
      )
    )
  )) AS sentiment
FROM cx.cx_silver.surveys
WHERE open_comment IS NOT NULL AND open_comment != '';

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM cx.cx_gold.survey_themes LIMIT 10;

survey_id,customer_id,survey_date,nps_score,nps_bucket,open_comment,theme,sentiment
SVY_0000001,CUST_008190,2025-11-03,4,Detractor,Software updates have introduced new bugs. Reliability is a major concern.,reliability,negative
SVY_0000002,CUST_008190,2025-10-20,1,Detractor,Too expensive for what we get. Considering switching vendors.,pricing,negative
SVY_0000003,CUST_037584,2026-01-24,5,Detractor,Too expensive for what we get. Considering switching vendors.,pricing,negative
SVY_0000004,CUST_012579,2025-02-25,10,Promoter,Consumables are high quality and shipping is consistent.,reliability,positive
SVY_0000005,CUST_028769,2025-10-31,10,Promoter,Consumables are high quality and shipping is consistent.,reliability,positive
SVY_0000007,CUST_023904,2025-08-06,4,Detractor,Field service rep canceled twice. Lost a full day of testing.,reliability,negative
SVY_0000008,CUST_023904,2024-09-01,3,Detractor,Downtime cost us major contracts. Need a more reliable partner.,reliability,negative
SVY_0000009,CUST_023904,2025-10-01,0,Detractor,Downtime cost us major contracts. Need a more reliable partner.,reliability,negative
SVY_0000010,CUST_007421,2024-02-02,9,Promoter,Great value for the price. Would buy again.,pricing,positive
SVY_0000011,CUST_043784,2024-05-04,9,Promoter,Customer success manager is incredible - really understands our workflow.,support_quality,positive


In [0]:
%sql
SELECT theme, COUNT(*) AS n
FROM cx.cx_gold.survey_themes
GROUP BY theme
ORDER BY n DESC;

theme,n
support_quality,7310
reliability,5325
software,4683
pricing,3537
training,2529


In [0]:
#Clean theme labels
from pyspark.sql.functions import col, when, lower, trim, regexp_extract

valid_themes = ["reliability", "support_quality", "training", "pricing", "software", "other"]
valid_sentiments = ["positive", "negative", "neutral"]

themes_raw = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.survey_themes")

themes_clean = (
    themes_raw
    # Extract first word in case LLM returned a sentence
    .withColumn("theme", regexp_extract(lower(trim(col("theme"))), r"([a-z_]+)", 1))
    .withColumn("sentiment", regexp_extract(lower(trim(col("sentiment"))), r"([a-z]+)", 1))
    # Fallback any unknown labels to "other" / "neutral"
    .withColumn("theme", when(col("theme").isin(valid_themes), col("theme")).otherwise("other"))
    .withColumn("sentiment", when(col("sentiment").isin(valid_sentiments), col("sentiment")).otherwise("neutral"))
)

themes_clean.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(
    f"{CATALOG}.{SCHEMA_GOLD}.survey_themes"
)
print(f"✅ Cleaned themes saved: {themes_clean.count():,} rows")


✅ Cleaned themes saved: 23,384 rows


In [0]:
%sql
--Theme summary table
CREATE OR REPLACE TABLE cx.cx_gold.theme_summary AS
SELECT
  theme,
  COUNT(*) AS mentions,
  ROUND(AVG(nps_score), 2) AS avg_nps,
  SUM(CASE WHEN sentiment = 'negative' THEN 1 ELSE 0 END) AS negative_mentions,
  SUM(CASE WHEN sentiment = 'positive' THEN 1 ELSE 0 END) AS positive_mentions,
  ROUND(100.0 * SUM(CASE WHEN sentiment = 'negative' THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_negative
FROM cx.cx_gold.survey_themes
GROUP BY theme
ORDER BY mentions DESC;

num_affected_rows,num_inserted_rows


In [0]:
%sql
--Theme by segment
CREATE OR REPLACE TABLE cx.cx_gold.theme_by_segment AS
SELECT
  s.segment_name,
  t.theme,
  COUNT(*) AS mentions,
  ROUND(AVG(t.nps_score), 2) AS avg_nps,
  ROUND(100.0 * SUM(CASE WHEN t.sentiment = 'negative' THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_negative
FROM cx.cx_gold.survey_themes t
JOIN cx.cx_gold.customer_segments s USING (customer_id)
GROUP BY s.segment_name, t.theme
ORDER BY s.segment_name, mentions DESC;

num_affected_rows,num_inserted_rows


In [0]:
%sql
--Theme by risk band
CREATE OR REPLACE TABLE cx.cx_gold.theme_by_risk_band AS
SELECT
  p.risk_band,
  t.theme,
  COUNT(*) AS mentions,
  ROUND(AVG(t.nps_score), 2) AS avg_nps
FROM cx.cx_gold.survey_themes t
JOIN cx.cx_gold.customer_propensity p USING (customer_id)
GROUP BY p.risk_band, t.theme
ORDER BY p.risk_band, mentions DESC;

num_affected_rows,num_inserted_rows


In [0]:
%sql
--Quick view of top themes
SELECT theme, mentions, avg_nps, pct_negative
FROM cx.cx_gold.theme_summary
ORDER BY mentions DESC;

theme,mentions,avg_nps,pct_negative
support_quality,7310,7.39,25.8
reliability,5325,5.49,69.5
software,4683,7.46,19.0
pricing,3537,5.99,53.9
training,2529,6.76,37.2


In [0]:
%sql
--Which themes drive the most detractor signal?
SELECT theme, mentions, avg_nps, pct_negative,
       CASE
         WHEN avg_nps <= 6 THEN 'detractor-dominant'
         WHEN avg_nps <= 8 THEN 'passive-leaning'
         ELSE 'promoter-dominant'
       END AS nps_lean
FROM cx.cx_gold.theme_summary
ORDER BY avg_nps ASC;

theme,mentions,avg_nps,pct_negative,nps_lean
reliability,5325,5.49,69.5,detractor-dominant
pricing,3537,5.99,53.9,detractor-dominant
training,2529,6.76,37.2,passive-leaning
support_quality,7310,7.39,25.8,passive-leaning
software,4683,7.46,19.0,passive-leaning


In [0]:
%sql
SELECT * FROM cx.cx_gold.theme_by_segment ORDER BY mentions DESC LIMIT 20;

segment_name,theme,mentions,avg_nps,pct_negative
Strategic Champions,support_quality,4798,7.4,25.8
Strategic Champions,reliability,3529,5.49,69.3
Strategic Champions,software,3020,7.47,19.3
Strategic Champions,pricing,2301,6.03,53.6
Strategic Champions,training,1620,6.69,38.2
Dormant / Disengaged,support_quality,1340,7.43,25.4
Dormant / Disengaged,reliability,974,5.44,71.0
Dormant / Disengaged,software,906,7.47,18.3
At-Risk High-Touch,support_quality,807,7.25,27.1
Dormant / Disengaged,pricing,687,5.91,54.6


In [0]:
%sql
SELECT * FROM cx.cx_gold.theme_by_risk_band ORDER BY risk_band, mentions DESC;

risk_band,theme,mentions,avg_nps
Critical,support_quality,1731,7.01
Critical,reliability,1433,5.16
Critical,software,1119,7.34
Critical,pricing,978,5.65
Critical,training,642,6.39
High,support_quality,1829,7.39
High,reliability,1399,5.56
High,software,1138,7.37
High,pricing,905,5.96
High,training,628,6.83


In [0]:
%sql
SELECT theme, avg_nps, pct_negative,
       CASE
         WHEN avg_nps <= 6 THEN 'detractor-dominant'
         WHEN avg_nps <= 8 THEN 'passive-leaning'
         ELSE 'promoter-dominant'
       END AS nps_lean
FROM cx.cx_gold.theme_summary
ORDER BY avg_nps ASC

theme,avg_nps,pct_negative,nps_lean
reliability,5.49,69.5,detractor-dominant
pricing,5.99,53.9,detractor-dominant
training,6.76,37.2,passive-leaning
support_quality,7.39,25.8,passive-leaning
software,7.46,19.0,passive-leaning
